In [ ]:
pip install requests pyodbc

In [ ]:
import requests
import pyodbc
import subprocess
import os
import struct

In [ ]:
result = subprocess.run(
            ["terraform", "output", "-raw", "function_url"],
            cwd="terraform/",          # Change working directory
            capture_output=True,      # Capture stdout and stderr
            text=True,                # Decode output as string
            check=False               # Don't raise exception on non-zero exit
        )
FUNCTION_BASE_URL=f"https://{result.stdout.strip()}"
print(FUNCTION_BASE_URL)

In [ ]:
resp = requests.get(f"{FUNCTION_BASE_URL}/www/HttpExample")
print(resp.text)

In [ ]:
result = subprocess.run(
            ["terraform", "output", "-raw", "sql_server_name"],
            cwd="terraform/",          # Change working directory
            capture_output=True,      # Capture stdout and stderr
            text=True,                # Decode output as string
            check=False               # Don't raise exception on non-zero exit
        )
SERVER_NAME=result.stdout.strip()
print(SERVER_NAME)
result = subprocess.run(
            ["terraform", "output", "-raw", "sql_database_name"],
            cwd="terraform/",          # Change working directory
            capture_output=True,      # Capture stdout and stderr
            text=True,                # Decode output as string
            check=False               # Don't raise exception on non-zero exit
        )
DATABASE_NAME=result.stdout.strip()
print(DATABASE_NAME)
UID=os.environ.get("AZURE_CLIENT_ID")
print(UID)

In [ ]:
conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SERVER_NAME}.database.windows.net;"
    f"DATABASE={DATABASE_NAME};"
    f"UID={UID};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;Authentication=ActiveDirectoryMsi;"
)

print(f"connecting to {conn_str}")
with pyodbc.connect(conn_str) as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT TOP 5 name FROM sys.tables;")
    rows = cursor.fetchall()

    # Format results
    table_names = [row[0] for row in rows]
    print(table_names)


In [ ]:
def handle_datetimeoffset(dto_value):
    # ref: https://github.com/mkleehammer/pyodbc/issues/134#issuecomment-281739794
    tup = struct.unpack("<6hI2h", dto_value)  # e.g., (2017, 3, 16, 10, 35, 18, 0, -6, 0)
    tweaked = [tup[i] // 100 if i == 6 else tup[i] for i in range(len(tup))]
    return "{:04d}-{:02d}-{:02d} {:02d}:{:02d}:{:02d}.{:07d} {:+03d}:{:02d}".format(*tweaked)

conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SERVER_NAME}.database.windows.net;"
    f"DATABASE={DATABASE_NAME};"
    f"UID={UID};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;Authentication=ActiveDirectoryMsi;"
)

print(f"connecting to {conn_str}")
with pyodbc.connect(conn_str) as conn:
    conn.add_output_converter(-155, handle_datetimeoffset)
    cursor = conn.cursor()
    cursor.execute("SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE, CHARACTER_MAXIMUM_LENGTH FROM INFORMATION_SCHEMA.COLUMNS;")
    rows = cursor.fetchall()

    # Format results
    for row in rows:
        print(row)


In [ ]:
def handle_datetimeoffset(dto_value):
    # ref: https://github.com/mkleehammer/pyodbc/issues/134#issuecomment-281739794
    tup = struct.unpack("<6hI2h", dto_value)  # e.g., (2017, 3, 16, 10, 35, 18, 0, -6, 0)
    tweaked = [tup[i] // 100 if i == 6 else tup[i] for i in range(len(tup))]
    return "{:04d}-{:02d}-{:02d} {:02d}:{:02d}:{:02d}.{:07d} {:+03d}:{:02d}".format(*tweaked)

conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={SERVER_NAME}.database.windows.net;"
    f"DATABASE={DATABASE_NAME};"
    f"UID={UID};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;Authentication=ActiveDirectoryMsi;"
)

print(f"connecting to {conn_str}")
with pyodbc.connect(conn_str) as conn:
    conn.add_output_converter(-155, handle_datetimeoffset)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM \"User\";")
    rows = cursor.fetchall()

    # Format results
    for row in rows:
        print(row)